# Project Additional Materials — Spatial Dataset Testing

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:**  
1. First run `Spatial_dataset_prepare.ipynb` to generate the prepared spatial datasets.  
   All output files are saved in the `splits_mm/` folder, which must be at the same level as this notebook.  
2. Then run this notebook top to bottom in the same directory.  
   It loads the prepared datasets from `splits_mm/` and performs testing and validation.  
   
This notebook evaluates the spatial datasets by running model tests on the prepared subsets (`top30`, `bottom30`).  
The evaluation results are printed to the notebook output. 


## Step 0 — Import required libraries

In [ ]:
import json, pickle, os
import numpy as np, pandas as pd
from sklearn.metrics import r2_score, mean_squared_error

## Step 1 — Define functions

In [ ]:
DTYPE = np.dtype("float32")

# Load dataset split from memory-mapped files
def load_split(prefix):
    with open(f"splits_mm/{prefix}_meta.json", "r") as f:
        meta = json.load(f)
    rows, cols = int(meta["rows"]), int(meta["cols"])
    X = np.memmap(f"splits_mm/{prefix}_X.mm", mode="r", dtype=DTYPE, shape=(rows, cols))
    y = np.memmap(f"splits_mm/{prefix}_y.mm", mode="r", dtype=DTYPE, shape=(rows,))
    return np.asarray(X), np.asarray(y), meta.get("features", None)

# Evaluate model performance
def evaluate(model_path, X, y, label):
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    preds = model.predict(X)
    rmse = mean_squared_error(y, preds, squared=False)
    r2   = r2_score(y, preds)
    print(f"{label} - R²: {r2:.4f}, RMSE: {rmse:.4f}")
    return r2, rmse

# Run test for specified task type
def run_test(task_type, model_path):
    if task_type == "low":
        split = "bottom30"
    elif task_type == "high":
        split = "top30"
    else:
        raise ValueError("task_type must be 'low' or 'high'")
    # Load test split
    X_test, y_test, feats = load_split(split)
    print(f"[info] Loaded {split} as test set, shape={X_test.shape}")
    return evaluate(model_path, X_test, y_test, f"{os.path.basename(model_path)} [{split}]")


## Step 2 — Evaluate low→high CV models on HIGH subset
Run the tests on the **HIGH** subset using the low→high cross-validation models and **print** evaluation metrics.

In [7]:
run_test("high", "automl_low2high_cv.pkl")
run_test("high", "automl_rf_low2high_cv.pkl")
run_test("high", "automl_xgb_low2high_cv.pkl")
run_test("high", "automl_lgbm_low2high_cv.pkl")

[info] Loaded top30 as test set, shape=(11031722, 10)
automl_low2high_cv.pkl [top30] - R²: 0.8986, RMSE: 1.6710
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_rf_low2high_cv.pkl [top30] - R²: 0.9030, RMSE: 1.6339
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_xgb_low2high_cv.pkl [top30] - R²: 0.8962, RMSE: 1.6903
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_lgbm_low2high_cv.pkl [top30] - R²: 0.8409, RMSE: 2.0928


(0.840895784856358, 2.092821389037513)

## Step 3 — Evaluate low→high validation models on HIGH subset
Run the tests on the **HIGH** subset using the low→high validation models and **print** evaluation metrics.

In [8]:
run_test("high", "automl_low2high_val.pkl")
run_test("high", "automl_rf_low2high_val.pkl")
run_test("high", "automl_xgb_low2high_val.pkl")
run_test("high", "automl_lgbm_low2high_val.pkl")


[info] Loaded top30 as test set, shape=(11031722, 10)
automl_low2high_val.pkl [top30] - R²: 0.9234, RMSE: 1.4524
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_rf_low2high_val.pkl [top30] - R²: 0.9232, RMSE: 1.4537
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_xgb_low2high_val.pkl [top30] - R²: 0.9147, RMSE: 1.5326
[info] Loaded top30 as test set, shape=(11031722, 10)
automl_lgbm_low2high_val.pkl [top30] - R²: 0.9130, RMSE: 1.5473


(0.9130292095032044, 1.5473124900271458)

## Step 4 — Evaluate high→low CV models on LOW subset
Run the tests on the **LOW** subset using the high→low cross-validation models and **print** evaluation metrics.

In [3]:
run_test("low", "automl_high2low_cv.pkl")
run_test("low", "automl_rf_high2low_cv.pkl")
run_test("low", "automl_xgb_high2low_cv.pkl")
run_test("low", "automl_lgbm_high2low_cv.pkl")

[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_high2low_cv.pkl [bottom30] - R²: 0.9253, RMSE: 1.3663
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_rf_high2low_cv.pkl [bottom30] - R²: 0.9111, RMSE: 1.4909
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_xgb_high2low_cv.pkl [bottom30] - R²: 0.9159, RMSE: 1.4502
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_lgbm_high2low_cv.pkl [bottom30] - R²: 0.8904, RMSE: 1.6552


(0.890377785553137, 1.6552393750800327)

## Step 5 — Evaluate high→low validation models on LOW subset
Run the tests on the **LOW** subset using the high→low validation models and **print** evaluation metrics.


In [2]:
run_test("low", "automl_high2low_val.pkl")
run_test("low", "automl_rf_high2low_val.pkl")
run_test("low", "automl_xgb_high2low_val.pkl")
run_test("low", "automl_lgbm_high2low_val.pkl")

[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_high2low_val.pkl [bottom30] - R²: 0.9424, RMSE: 1.1995
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_rf_high2low_val.pkl [bottom30] - R²: 0.9424, RMSE: 1.1995
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_xgb_high2low_val.pkl [bottom30] - R²: 0.9297, RMSE: 1.3252
[info] Loaded bottom30 as test set, shape=(11031722, 10)
automl_lgbm_high2low_val.pkl [bottom30] - R²: 0.9353, RMSE: 1.2713


(0.9353334296822085, 1.27130992491364)

## Step 6 — Inspect feature importance (RF, high→low validation)
Load `automl_rf_high2low_val.pkl` and **print** Gini-based feature importances for the selected features.


In [ ]:
with open("automl_rf_high2low_val.pkl","rb") as f:
    model = pickle.load(f)

feature_names = ["FLNS","FSNS","PRECT","PRSN","QBOT","TREFHT","UBOT","VBOT","lat","lon"]
imp = np.asarray(model.feature_importances_, dtype=float)

df = (pd.DataFrame({"feature": feature_names, "gini": imp, "gini_%": imp/imp.sum()*100})
        .sort_values("gini", ascending=False).reset_index(drop=True))
print("Feature importance (high training):")
print(df.to_string(index=False))

Feature importance (high training):
feature     gini    gini_%
 TREFHT 0.512490 51.249017
   QBOT 0.271473 27.147256
   FSNS 0.149678 14.967798
   FLNS 0.021112  2.111167
   UBOT 0.016333  1.633298
    lat 0.013358  1.335770
   VBOT 0.010974  1.097406
    lon 0.004574  0.457393
   PRSN 0.000005  0.000499
  PRECT 0.000004  0.000396


## Step 7 — Inspect feature importance (RF, low→high validation)
Load `automl_rf_low2high_val.pkl` and **print** Gini-based feature importances for the selected features.


In [13]:
with open("automl_rf_low2high_val.pkl","rb") as f:
    model = pickle.load(f)

feature_names = ["FLNS","FSNS","PRECT","PRSN","QBOT","TREFHT","UBOT","VBOT","lat","lon"]
imp = np.asarray(model.feature_importances_, dtype=float)

df = (pd.DataFrame({"feature": feature_names, "gini": imp, "gini_%": imp/imp.sum()*100})
        .sort_values("gini", ascending=False).reset_index(drop=True))
print("Feature importance (low training):")
print(df.to_string(index=False))

Feature importance (low training):
feature         gini    gini_%
 TREFHT 4.923181e-01 49.231808
   QBOT 2.984616e-01 29.846156
   FSNS 1.461810e-01 14.618102
   FLNS 2.216153e-02  2.216153
   UBOT 1.237653e-02  1.237653
    lon 1.173234e-02  1.173234
   VBOT 1.032196e-02  1.032196
    lat 6.442786e-03  0.644279
  PRECT 3.575116e-06  0.000358
   PRSN 6.127299e-07  0.000061


## Step 8 — Inspect feature importance (RF, high→low CV)
Load `automl_rf_high2low_cv.pkl` and **print** Gini-based feature importances for the selected features.


In [14]:
with open("automl_rf_high2low_cv.pkl","rb") as f:
    model = pickle.load(f)

feature_names = ["FLNS","FSNS","PRECT","PRSN","QBOT","TREFHT","UBOT","VBOT","lat","lon"]
imp = np.asarray(model.feature_importances_, dtype=float)

df = (pd.DataFrame({"feature": feature_names, "gini": imp, "gini_%": imp/imp.sum()*100})
        .sort_values("gini", ascending=False).reset_index(drop=True))
print("Feature importance (high training):")
print(df.to_string(index=False))

Feature importance (high training):
feature     gini    gini_%
 TREFHT 0.965838 96.583804
   FSNS 0.025306  2.530635
    lat 0.008856  0.885561
   FLNS 0.000000  0.000000
  PRECT 0.000000  0.000000
   PRSN 0.000000  0.000000
   QBOT 0.000000  0.000000
   UBOT 0.000000  0.000000
   VBOT 0.000000  0.000000
    lon 0.000000  0.000000


## Step 9 — Inspect feature importance (RF, low→high CV)
Load `automl_rf_low2high_cv.pkl` and **print** Gini-based feature importances for the selected features.

In [17]:
with open("automl_rf_low2high_cv.pkl","rb") as f:
    model = pickle.load(f)

feature_names = ["FLNS","FSNS","PRECT","PRSN","QBOT","TREFHT","UBOT","VBOT","lat","lon"]
imp = np.asarray(model.feature_importances_, dtype=float)

df = (pd.DataFrame({"feature": feature_names, "gini": imp, "gini_%": imp/imp.sum()*100})
        .sort_values("gini", ascending=False).reset_index(drop=True))
print("Feature importance (low training):")
print(df.to_string(index=False))

Feature importance (low training):
feature     gini    gini_%
 TREFHT 0.972213 97.221292
   FSNS 0.021624  2.162355
    lon 0.004281  0.428148
   FLNS 0.001882  0.188204
  PRECT 0.000000  0.000000
   PRSN 0.000000  0.000000
   QBOT 0.000000  0.000000
   UBOT 0.000000  0.000000
   VBOT 0.000000  0.000000
    lat 0.000000  0.000000
